# 基于rating2inter.ipynb生成的5-core交互图，Train/Validation/Test data splitting
- Based on generated interactions, perform data splitting


In [1]:
import os, csv
import pandas as pd

In [2]:
!ls

0rating2inter.ipynb  2reindex-feat.ipynb  dualgnn-gen-u-u-matrix.py
1splitting.ipynb     3feat-encoder.ipynb  README.md


In [3]:
os.chdir('../data/grocery')
os.getcwd()

'/home/hun/MMRec_AlignRec/data/grocery'

## 直接加载现成的, Load interactions

In [4]:
rslt_file = 'grocery-indexed.inter'
df = pd.read_csv(rslt_file, sep='\t')
print(f'shape: {df.shape}')
df[:4]

shape: (151254, 5)


,userID,itemID,rating,timestamp,x_label
0,0,0,4.0,1370044800,0
1,2,0,4.0,1381190400,0
2,3,0,5.0,1369008000,0
3,4,0,4.0,1369526400,0


In [5]:
import random
import numpy as np

In [6]:

df = df.sample(frac=1).reset_index(drop=True)

df.sort_values(by=['userID'], inplace=True)
df[:20]

,userID,itemID,rating,timestamp,x_label
143879,0,7129,5.0,1391558400,1
21573,0,2204,3.0,1377820800,0
58185,0,0,4.0,1370044800,0
42402,0,4961,5.0,1345334400,0
114698,0,1115,5.0,1362700800,0
134711,0,6151,4.0,1373587200,0
62253,0,391,4.0,1356307200,0
72440,1,2485,5.0,1384905600,0
8054,1,7271,4.0,1355788800,0
146500,1,819,5.0,1367798400,0


In [7]:
uid_field, iid_field = 'userID', 'itemID'

uid_freq = df.groupby(uid_field)[iid_field]
u_i_dict = {}
for u, u_ls in uid_freq:
    u_i_dict[u] = list(u_ls)
u_i_dict

{0: [7129, 2204, 0, 4961, 1115, 6151, 391],
 1: [2485,
  7271,
  819,
  6600,
  5363,
  4913,
  5179,
  0,
  5290,
  5178,
  8498,
  4756,
  3533,
  7024,
  5294,
  5015,
  7260,
  7489,
  8639,
  6191,
  3968,
  5292,
  3073,
  323,
  3343,
  3048],
 2: [6190, 2124, 6630, 193, 4430, 0],
 3: [6186, 0, 6450, 7266, 240, 8178, 7664, 5886],
 4: [7224, 2162, 5277, 0, 6571, 2004, 326],
 5: [0, 7821, 7625, 5341, 3685],
 6: [0, 323, 2313, 1266, 897, 3966],
 7: [3640, 7324, 1738, 6140, 0, 2060],
 8: [6403, 7632, 0, 5240, 5754],
 9: [5521, 8101, 4061, 236, 0, 5565, 6045],
 10: [5504, 577, 0, 4199, 5713, 2408, 5170],
 11: [2321, 493, 6322, 4807, 0],
 12: [4690, 0, 723, 7269, 1936, 1958, 3955, 1654, 1792, 4847, 5897, 8146],
 13: [4896, 858, 0, 2014, 4905],
 14: [6403, 1010, 3066, 1467, 1662, 0],
 15: [4353, 308, 3096, 4951, 0, 6879, 675, 921],
 16: [299, 6754, 7787, 3371, 1],
 17: [1974, 1936, 6737, 1, 1750, 437],
 18: [2094,
  775,
  6381,
  6356,
  687,
  7072,
  7152,
  7478,
  7175,
  968,
  1

In [8]:
new_label = []
u_ids_sorted = sorted(u_i_dict.keys())

for u in u_ids_sorted:
    items = u_i_dict[u]
    n_items = len(items)
    if n_items < 10:
        tmp_ls = [0] * (n_items - 2) + [1] + [2]
    else:
        val_test_len = int(n_items * 0.2)
        train_len = n_items - val_test_len
        val_len = val_test_len // 2
        test_len = val_test_len - val_len
        tmp_ls = [0] * train_len + [1] * val_len + [2] * test_len
    new_label.extend(tmp_ls)

new_label[:100]

[0,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 2,
 2,
 2,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0]

In [9]:
df['x_label'] = new_label
df[:20]

,userID,itemID,rating,timestamp,x_label
143879,0,7129,5.0,1391558400,0
21573,0,2204,3.0,1377820800,0
58185,0,0,4.0,1370044800,0
42402,0,4961,5.0,1345334400,0
114698,0,1115,5.0,1362700800,0
134711,0,6151,4.0,1373587200,1
62253,0,391,4.0,1356307200,2
72440,1,2485,5.0,1384905600,0
8054,1,7271,4.0,1355788800,0
146500,1,819,5.0,1367798400,0


In [10]:
rslt_file[:-6]

'grocery-indexed'

In [11]:
new_labeled_file = rslt_file[:-6] + '-v4.inter'
df.to_csv(os.path.join('./', new_labeled_file), sep='\t', index=False)
print('done!!!')

done!!!


## Reload

In [12]:
indexed_df = pd.read_csv(new_labeled_file, sep='\t')
print(f'shape: {indexed_df.shape}')
indexed_df[:20]

shape: (151254, 5)


,userID,itemID,rating,timestamp,x_label
0,0,7129,5.0,1391558400,0
1,0,2204,3.0,1377820800,0
2,0,0,4.0,1370044800,0
3,0,4961,5.0,1345334400,0
4,0,1115,5.0,1362700800,0
5,0,6151,4.0,1373587200,1
6,0,391,4.0,1356307200,2
7,1,2485,5.0,1384905600,0
8,1,7271,4.0,1355788800,0
9,1,819,5.0,1367798400,0


In [13]:
u_id_str, i_id_str = 'userID', 'itemID'
u_uni = indexed_df[u_id_str].unique()
c_uni = indexed_df[i_id_str].unique()

print(f'# of unique learners: {len(u_uni)}')
print(f'# of unique courses: {len(c_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(u_uni), max(u_uni)))
print('min/max of unique courses: {0}/{1}'.format(min(c_uni), max(c_uni)))


# of unique learners: 14681
# of unique courses: 8713
min/max of unique learners: 0/14680
min/max of unique courses: 0/8712
